# RBC Bounding Boxes + Contours (Separate Outputs)

Uses the box-only `single_cls` YOLO model to get RBC bounding boxes, then one
whole-image marker-based **watershed** pass (seeded with each box's own
center) to split every cell out at its real pixel boundary -- including
touching/overlapping RBCs, which a per-box threshold can't separate since two
touching same-stain cells show no visible edge between them.

**Boxes and contours are saved as two SEPARATE overlay images per input FOV**
(not combined into one) — `overlays_boxes/` and `overlays_contours/`.

In [ ]:
import json, time
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

# ── Config ────────────────────────────────────────────────────────────────────
BEST_PT     = Path('F:/Livo/Data - 2026/Rbc/rbc_yolo/yolo11n_1024_singlecls/weights/best.pt')
IMAGES_DIR  = Path(r'F:\Livo\Data - 2026\1753802296\testdata')
OUT_DIR     = Path('F:/Livo/Data - 2026/Rbc/rbc_contour_output_1')
N_IMAGES    = 10          # None = run all images in IMAGES_DIR

IMGSZ       = 1024
CONF_THRES  = 0.10
IOU_THRES   = 0.7
SEED_FRAC   = 0.15        # watershed seed radius, as a fraction of each box's
                           # shorter side
CROP_PAD    = 25         
BOX_EXCLUDE_PAD = 8       
MIN_AREA_FRAC = 0.2      
SMOOTH_WINDOW = 9         # circular moving-average window used to smooth the
                           # pixel staircase cv2.findContours produces
SIMPLIFY_EPS  = 1.2     
SEED_CONTAINMENT_THRESH = 0.7   
BOX_COLOR   = (0, 255, 0)
CONTOUR_COLOR = (0, 255, 0)

print(f'BEST_PT: {BEST_PT}  [{"EXISTS" if BEST_PT.exists() else "NOT FOUND"}]')

In [2]:
_CLOSE_KERNEL = np.ones((3, 3), np.uint8)


def _smooth_closed_contour(pts, window=SMOOTH_WINDOW):
    """Circular moving-average over a closed contour's points. cv2.findContours
    traces the mask pixel-by-pixel, so its raw output is a staircase even for
    a genuinely round cell; averaging each point with its neighbors (wrapping
    around, since the contour is a loop) turns that staircase into a smooth
    curve without changing the cell's actual shape."""
    n = len(pts)
    if n < window * 2:
        return pts
    pad = window // 2
    ext = np.concatenate([pts[-pad:], pts, pts[:pad]], axis=0)
    kernel = np.ones(window, dtype=np.float32) / window
    xs = np.convolve(ext[:, 0], kernel, mode='valid')
    ys = np.convolve(ext[:, 1], kernel, mode='valid')
    return np.stack([xs, ys], axis=1).astype(np.float32)


def _watershed_markers(img_bgr, boxes):
    """Background = label 1, box i's seed = label i+2, everything else
    (unknown) = 0 for cv2.watershed to resolve. Background is only ever
    marked OUTSIDE every (padded) YOLO box -- never from raw intensity alone
    -- so a cell's own pale/hypochromic center can't get misread as
    background and wall its seed in before it floods out to the real edge."""
    H, W = img_bgr.shape[:2]
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, fg = cv2.threshold(hsv[:, :, 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, _CLOSE_KERNEL)
    fg = cv2.dilate(fg, _CLOSE_KERNEL, iterations=2)

    any_box = np.zeros((H, W), dtype=np.uint8)
    for x1, y1, x2, y2 in boxes:
        bx1, by1 = max(0, x1 - BOX_EXCLUDE_PAD), max(0, y1 - BOX_EXCLUDE_PAD)
        bx2, by2 = min(W, x2 + BOX_EXCLUDE_PAD), min(H, y2 + BOX_EXCLUDE_PAD)
        any_box[by1:by2, bx1:bx2] = 255

    markers = np.zeros((H, W), dtype=np.int32)
    markers[(fg == 0) & (any_box == 0)] = 1
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        r = max(2, int(SEED_FRAC * min(x2 - x1, y2 - y1)))
        cv2.circle(markers, (cx, cy), r, i + 2, -1)
    return markers


def _dedup_for_seeding(boxes, confidences, containment_thresh=SEED_CONTAINMENT_THRESH):
    """Suppress a box from getting its own watershed seed when it's almost
    entirely nested inside another, higher-confidence box -- a near-duplicate
    detection of the same physical cell. Standard NMS (by IoU) can miss this:
    a small box fully inside a much larger one can have moderate IoU (skewed
    by the size difference) even though it's clearly redundant. Suppressed
    boxes keep their bounding box for recall -- they just get no seed/contour."""
    order = sorted(range(len(boxes)), key=lambda i: -confidences[i])
    keep = [True] * len(boxes)
    for ai in range(len(order)):
        ia = order[ai]
        if not keep[ia]:
            continue
        ax1, ay1, ax2, ay2 = boxes[ia]
        area_a = (ax2 - ax1) * (ay2 - ay1)
        for bi in range(ai + 1, len(order)):
            ib = order[bi]
            if not keep[ib]:
                continue
            bx1, by1, bx2, by2 = boxes[ib]
            ix1, iy1 = max(ax1, bx1), max(ay1, by1)
            ix2, iy2 = min(ax2, bx2), min(ay2, by2)
            iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
            inter = iw * ih
            area_b = (bx2 - bx1) * (by2 - by1)
            smaller = min(area_a, area_b)
            if smaller and inter / smaller > containment_thresh:
                keep[ib] = False
    return keep


def watershed_contours(img_bgr, boxes, confidences):
    """One global watershed pass over the whole FOV, seeded with each YOLO
    box's own center as a distinct marker (near-duplicate boxes are excluded
    from seeding via _dedup_for_seeding first). Unlike per-box thresholding,
    watershed floods outward from every seed at once and draws the split
    boundary exactly where two floods meet -- so touching/overlapping
    same-colored RBCs get split at their real shared edge instead of coming
    out as one merged blob or a box-shaped notch. Each contour is then
    smoothed (removing the raw pixel staircase) and simplified back down to a
    compact polygon. Returns a list the same length as boxes; each entry is
    an (N,2) float32 contour in ORIGINAL image coordinates, or None if that
    box's region came out empty/degenerate or was excluded from seeding."""
    H, W = img_bgr.shape[:2]
    keep = _dedup_for_seeding(boxes, confidences)
    seed_indices = [i for i, k in enumerate(keep) if k]
    seed_boxes = [boxes[i] for i in seed_indices]
    markers = _watershed_markers(img_bgr, seed_boxes)
    cv2.watershed(img_bgr, markers)   # mutates markers in place; ridges -> -1

    out = [None] * len(boxes)
    for pos, orig_i in enumerate(seed_indices):
        x1, y1, x2, y2 = boxes[orig_i]
        label = pos + 2
        cx1, cy1 = max(0, x1 - CROP_PAD), max(0, y1 - CROP_PAD)
        cx2, cy2 = min(W, x2 + CROP_PAD), min(H, y2 + CROP_PAD)
        region = (markers[cy1:cy2, cx1:cx2] == label).astype(np.uint8) * 255
        if not region.any():
            continue
        # CHAIN_APPROX_NONE (every boundary pixel, not just corners) so the
        # smoothing step below has enough points to average over
        contours, _ = cv2.findContours(region, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not contours:
            continue
        best = max(contours, key=cv2.contourArea)
        box_area = (x2 - x1) * (y2 - y1)
        if cv2.contourArea(best) < MIN_AREA_FRAC * box_area:
            continue
        pts = best.reshape(-1, 2).astype(np.float32)
        pts[:, 0] += cx1   # back to original image coordinates
        pts[:, 1] += cy1
        smoothed = _smooth_closed_contour(pts)
        simplified = cv2.approxPolyDP(smoothed.reshape(-1, 1, 2), SIMPLIFY_EPS, True)
        out[orig_i] = simplified.reshape(-1, 2)
    return out


def detect_and_contour(model, img_bgr):
    """Runs YOLO box detection, then one whole-image watershed pass for all
    boxes together. Returns (detections, timing_dict)."""
    H, W = img_bgr.shape[:2]

    t0 = time.perf_counter()
    results = model.predict(img_bgr, conf=CONF_THRES, iou=IOU_THRES, imgsz=IMGSZ, verbose=False)
    detect_ms = (time.perf_counter() - t0) * 1000

    t1 = time.perf_counter()
    boxes, confidences = [], []
    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        for box in r.boxes:
            x1, y1, x2, y2 = [round(float(v)) for v in box.xyxy[0]]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            if x2 - x1 <= 0 or y2 - y1 <= 0:
                continue
            boxes.append((x1, y1, x2, y2))
            confidences.append(round(float(box.conf[0]), 4))

    contours = watershed_contours(img_bgr, boxes, confidences)

    dets = []
    for det_id, ((x1, y1, x2, y2), conf, pts) in enumerate(zip(boxes, confidences, contours), start=1):
        det = {
            'id':         det_id,
            'bbox':       {'x': x1, 'y': y1, 'width': x2 - x1, 'height': y2 - y1},
            'confidence': conf,
        }
        if pts is not None:
            det['contour'] = [[round(float(px), 1), round(float(py), 1)] for px, py in pts]
        dets.append(det)
    contour_ms = (time.perf_counter() - t1) * 1000

    timing = {'detect_ms': round(detect_ms, 2), 'contour_ms': round(contour_ms, 2),
              'total_ms': round(detect_ms + contour_ms, 2)}
    return dets, timing


def draw_boxes(img_bgr, detections):
    """Bounding boxes ONLY."""
    vis = img_bgr.copy()
    for det in detections:
        b = det['bbox']
        cv2.rectangle(vis, (b['x'], b['y']), (b['x'] + b['width'], b['y'] + b['height']), BOX_COLOR, 1)
    return vis


def draw_contours(img_bgr, detections):
    """Contours ONLY (falls back to nothing drawn for a detection with no contour)."""
    vis = img_bgr.copy()
    for det in detections:
        if 'contour' in det:
            pts = np.array(det['contour'], dtype=np.int32).reshape(-1, 1, 2)
            cv2.polylines(vis, [pts], isClosed=True, color=CONTOUR_COLOR, thickness=1, lineType=cv2.LINE_AA)
    return vis


print('Helpers ready: watershed_contours, detect_and_contour, draw_boxes, draw_contours')

NameError: name 'SMOOTH_WINDOW' is not defined

In [ ]:
model = YOLO(str(BEST_PT))
print(f'Loaded: {BEST_PT}')

Loaded: F:\Livo\Data - 2026\Rbc\rbc_yolo\yolo11n_1024_singlecls\weights\best.pt


In [ ]:
BOX_OVERLAY_DIR     = OUT_DIR / 'overlays_boxes'
CONTOUR_OVERLAY_DIR = OUT_DIR / 'overlays_contours'
JSON_OUT            = OUT_DIR / 'json'
BOX_OVERLAY_DIR.mkdir(parents=True, exist_ok=True)
CONTOUR_OVERLAY_DIR.mkdir(parents=True, exist_ok=True)
JSON_OUT.mkdir(parents=True, exist_ok=True)

images = sorted(IMAGES_DIR.glob('*.jpg'))
if N_IMAGES is not None:
    images = images[:N_IMAGES]
print(f'Running on {len(images)} images -> {OUT_DIR}\n')

for idx, img_path in enumerate(images, 1):
    img = cv2.imread(str(img_path))
    if img is None:
        print(f'  [{idx:3d}] SKIP (unreadable): {img_path.name}')
        continue

    dets, timing = detect_and_contour(model, img)
    n_contours = sum(1 for d in dets if 'contour' in d)

    box_vis = draw_boxes(img, dets)
    cv2.imwrite(str(BOX_OVERLAY_DIR / (img_path.stem + '_boxes.jpg')), box_vis)

    contour_vis = draw_contours(img, dets)
    cv2.imwrite(str(CONTOUR_OVERLAY_DIR / (img_path.stem + '_contours.jpg')), contour_vis)

    with open(JSON_OUT / (img_path.stem + '_pred.json'), 'w') as f:
        json.dump({'image_id': img_path.name, 'detections': dets, 'timing': timing}, f, indent=2)

    print(f"  [{idx:3d}/{len(images)}] {img_path.name:<22s}  {len(dets):3d} boxes  "
          f"{n_contours:3d} contours  detect={timing['detect_ms']:6.1f}ms  "
          f"contour={timing['contour_ms']:6.1f}ms  total={timing['total_ms']:6.1f}ms")

print(f'\nBox overlays     -> {BOX_OVERLAY_DIR}')
print(f'Contour overlays -> {CONTOUR_OVERLAY_DIR}')
print(f'JSONs            -> {JSON_OUT}')

Running on 20 images -> F:\Livo\Data - 2026\Rbc\rbc_contour_output

  [  1/20] Img_8_0.jpg             262 boxes  248 contours  detect= 569.5ms  contour= 157.5ms  total= 727.0ms
  [  2/20] Img_8_1.jpg             247 boxes  219 contours  detect=  14.6ms  contour= 170.0ms  total= 184.6ms
  [  3/20] Img_8_10.jpg            254 boxes  235 contours  detect=  15.6ms  contour= 140.8ms  total= 156.4ms
  [  4/20] Img_8_11.jpg            243 boxes  223 contours  detect=  16.0ms  contour= 143.6ms  total= 159.5ms
  [  5/20] Img_8_12.jpg            228 boxes  219 contours  detect=  14.1ms  contour= 139.1ms  total= 153.2ms
  [  6/20] Img_8_13.jpg            252 boxes  235 contours  detect=  15.4ms  contour= 148.8ms  total= 164.2ms
  [  7/20] Img_8_14.jpg            231 boxes  227 contours  detect=  14.9ms  contour= 148.5ms  total= 163.4ms
  [  8/20] Img_8_15.jpg            232 boxes  224 contours  detect=  17.8ms  contour= 149.6ms  total= 167.3ms
  [  9/20] Img_8_16.jpg            238 boxes  218 co